# Working with Ontology-Generated Classes

The `semantic_objects.s223` classes are no longer hand-written - they're generated
directly from the real ASHRAE 223P ontology (vendored at
`src/semantic_objects/ontologies/s223/223p.ttl`), by walking its SHACL shapes. This
notebook shows how to use the result: instantiate objects, generate SPARQL queries,
SHACL shapes, and BuildingMOTIF templates.

If you want to see *how* the generation itself works (parsing the ontology, the
generated-vs-hand-written split, extending it), see
`ontology-ingestion-tutorial.ipynb`. This notebook is about using what comes out of
that pipeline.

In [1]:
import logging
logging.disable(logging.WARNING)  # buildingmotif is chatty at DEBUG/INFO

from pprint import pprint
from semantic_objects.s223 import entities, properties, enumerationkinds, relations
from semantic_objects.build_model import BMotifSession

print("Imports OK")

CRITICAL:root:Install the 'bacnet-ingress' module, e.g. 'pip install buildingmotif[bacnet-ingress]'


Imports OK


## 1. Explore a generated class

Every class under `semantic_objects.s223.entities`/`properties`/`enumerationkinds` is
a real dataclass with fields, relations, and cardinality metadata pulled straight from
the ontology's SHACL shapes.

In [2]:
for field_name, field_obj in entities.DomainSpace.__dataclass_fields__.items():
    print(f"{field_name}: {field_obj.type.__name__}  "
          f"(relation={field_obj.metadata['relation'].__name__}, "
          f"min={field_obj.metadata['min']}, max={field_obj.metadata['max']})")

domain: Domain  (relation=hasDomain, min=1, max=1)


In [3]:
print(entities.DomainSpace.comment)

A portion of a `PhysicalSpace` that is affected by a building service associated with a domain. `DomainSpace`s may represent an entire `PhysicalSpace` or any portion of a `PhysicalSpace` (see {s223:PhysicalSpace}). Multiple `DomainSpace`s of the same domain may overlap, and `DomainSpace`s of different domains may also overlap, but `DomainSpace`s can not overlap multiple `PhysicalSpace`s. `DomainSpace`s may be grouped into `Zone`s using the relation `hasDomainSpace` (see {s223:hasDomainSpace}).


## 2. Instantiate objects

`DomainSpace.domain` requires an `EnumerationKind-Domain` value - the ontology defines
a fixed set of these (HVAC, Electrical, Plumbing, ...).

In [4]:
print([n for n in dir(enumerationkinds)
       if isinstance(getattr(enumerationkinds, n), type)
       and issubclass(getattr(enumerationkinds, n), enumerationkinds.Domain)
       and getattr(enumerationkinds, n) is not enumerationkinds.Domain])

['ConveyanceSystems', 'Electrical', 'FireProtection', 'HVAC', 'Lighting', 'Networking', 'Occupancy', 'PhysicalSecurity', 'Plumbing', 'Refrigeration']


In [5]:
# The quick way: instantiate DomainSpace directly, passing an
# already-built EnumerationKind value for `domain`.
zone = entities.DomainSpace(domain=enumerationkinds.HVAC())
zone._name = "Zone_101"
print(zone._name, "->", zone.domain._name)

Zone_101 -> Domain-HVAC


### A reusable subclass instead

If you're going to create many `DomainSpace`s that are always HVAC zones, it's
more convenient to fix `domain` once on a subclass rather than passing it in
every time. Plain Python subclassing isn't enough on its own - re-apply
`@semantic_object` so the dataclass machinery re-runs and picks up the
overridden field as a *fixed value* instead of a required constructor argument.

`_semantic_type` tells the template/SPARQL/SHACL generators which real ontology
class this subclass stands in for. Without it, the generated templates would
assert `a s223:hvac_zone` - a class the ontology doesn't know about. With it,
they assert `a s223:DomainSpace` instead, with `hasDomain` baked in as a fixed
triple. (The same mechanism is used for `Area`/`QuantifiableObservableProperty`
in section 8 below.)

In [6]:
from semantic_objects.core import semantic_object

@semantic_object
class hvac_zone(entities.DomainSpace):
    domain = enumerationkinds.HVAC()
    _semantic_type = entities.DomainSpace

In [7]:
print(hvac_zone.to_yaml())

hvac_zone:
  body: >+
    @prefix P: <urn:___param___#> .

    @prefix s223: <http://data.ashrae.org/standard223#> .


    P:name a s223:DomainSpace ;
        s223:hasDomain s223:Domain-HVAC .

  dependencies: []



In [8]:
# domain is now fixed - instances only need a _name
zone = hvac_zone()
zone._name = "Zone_101"
print(zone._name, "->", zone.domain._name)

Zone_101 -> Domain-HVAC


### Qualified fields: `Pump`'s connection points

`Pump` demonstrates a richer pattern. In the real ontology, a Pump's inlet and outlet
are both modeled via the same `hasConnectionPoint` relation, but SHACL narrows each to
a specific subtype (`InletConnectionPoint` / `OutletConnectionPoint`) using a
*qualified value shape*. The ingestion pipeline turns each of those into its own typed
field - so `Pump` gets `outlet_connection_point` and `inlet_connection_point` fields,
the same way `DomainSpace` gets `domain`.

In [9]:
water = enumerationkinds.Water()
connection = entities.Connection(medium=water)
outlet = entities.OutletConnectionPoint(medium=water, connection=connection)
inlet = entities.InletConnectionPoint(medium=water, connection=connection)

pump = entities.Pump(outlet_connection_point=outlet, inlet_connection_point=inlet)
print(pump._name, "outlet medium:", pump.outlet_connection_point.medium._name)

Pump_1 outlet medium: Fluid-Water


In [10]:
dir(entities.Pump)

['__annotations__',
 '__class__',
 '__dataclass_fields__',
 '__dataclass_params__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__match_args__',
 '__module__',
 '__ne__',
 '__new__',
 '__post_init__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__sizeof__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 '_create_qualified_value_shape',
 '_get_attributes',
 '_get_evaluation_dict',
 '_get_inter_field_relations',
 '_get_iri',
 '_get_template_parameters',
 '_infer_relation_for_field',
 '_instance_counter',
 '_inter_field_relations',
 '_local_name',
 '_name',
 '_ns',
 '_other_types',
 '_resolve_fixed_default',
 '_valid_relations',
 'abstract',
 'comment',
 'generate_rdf_class_definition',
 'generate_turtle_body',
 'generate_yaml_template',
 'get_dependencies',
 'get_field_values',
 'get_optional_fields',

## 3. Generate a SPARQL query

`get_sparql_query()` walks the class's fields and their relations to build a query
that finds matching instances in an RDF graph.

In [11]:
print(entities.DomainSpace.get_sparql_query(ontology='s223'))

PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX s223: <http://data.ashrae.org/standard223#>
SELECT DISTINCT * WHERE { ?domain rdf:type s223:EnumerationKind-Domain .
?name s223:hasDomain ?domain .
?name rdf:type s223:DomainSpace . }


## 4. Generate SHACL

`generate_rdf_class_definition()` re-derives a SHACL shape from the Python class -
useful for validating instance data, or for round-tripping back toward the ontology.

In [12]:
print(entities.Pump.generate_rdf_class_definition())

@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix s223: <http://data.ashrae.org/standard223#> .
@prefix sh: <http://www.w3.org/ns/shacl#> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

s223:Pump a s223:Class,
        rdfs:Class,
        sh:NodeShape ;
    rdfs:label "Pump" ;
    rdfs:comment "A piece of `Equipment` that imparts energy to a fluid, drawing a fluid into itself through an inlet port, and forcing the fluid out through an outlet port." ;
    rdfs:subClassOf s223:Equipment ;
    sh:property [ a sh:PropertyShape ;
            rdfs:comment "If the relation `hasConnectionPoint` is present it must associate the `Pump` with a `OutletConnectionPoint`." ;
            sh:minCount 2 ;
            sh:path s223:hasConnectionPoint ;
            sh:qualifiedMinCount 1 ;
            sh:qualifiedValueShape [ a sh:NodeShape ;
                    rdfs:label "None" ;
                    sh:class s223:InletConnectionPoint ],
                [ a sh:NodeShape ;
              

## 5. Generate a BuildingMOTIF template

`to_yaml()` emits a BuildingMOTIF-format template (Turtle body + dependencies).

In [13]:
print(entities.DomainSpace.to_yaml())

DomainSpace:
  body: >+
    @prefix P: <urn:___param___#> .

    @prefix s223: <http://data.ashrae.org/standard223#> .


    P:name a s223:DomainSpace ;
        s223:hasDomain P:domain .

  dependencies:
  - args:
      name: domain
    template: Domain



## 6. Load templates and evaluate an instance with BuildingMOTIF

`BMotifSession` wires the generated templates into an actual BuildingMOTIF model and
evaluates a real object against them, producing an RDF graph.

In [14]:
session = BMotifSession(ns='tutorial')
session.evaluate(zone)
print("Templates loaded:", list(session.templates.keys()))
print(session.model.graph.serialize(format='turtle'))

{'name': rdflib.term.URIRef('urn:tutorial#Zone_101'), 'domain': rdflib.term.URIRef('urn:tutorial#Domain-HVAC'), 'domain-_type': rdflib.term.Literal('HVAC')}
Templates loaded: ['hasDomain', 'Domain', 'hvac_zone']
@prefix ns1: <http://data.ashrae.org/standard223#> .
@prefix owl: <http://www.w3.org/2002/07/owl#> .

<urn:tutorial#> a owl:Ontology .

<urn:tutorial#Zone_101> a ns1:DomainSpace ;
    ns1:hasDomain ns1:Domain-HVAC .




## 7. What's *not* a field: complex SHACL constraints

Not every SHACL constraint maps to a Python field. `Pump`'s connection points also
carry a medium-compatibility rule (e.g. "the outlet must use water, oil, or
refrigerant") and a cross-instance check comparing all of a Pump's connection points
to each other - neither is representable as a single typed field. These are preserved,
not dropped, in `_raw_shapes.RAW_SHAPES`, keyed by class name.

In [15]:
from semantic_objects.s223._generated import _raw_shapes

for entry in _raw_shapes.RAW_SHAPES['Pump']:
    print(f"[{entry['kind']}] field={entry.get('field_name')}: {entry.get('message') or entry.get('comment')}")

[nested-node] field=outlet_connection_point: s223: A `Pump` shall have at least one outlet using the medium `Fluid-Water`, `Fluid-Oil` or `Constituent-Refrigerant`.
[sparql] field=outlet_connection_point: The non-electrical `ConnectionPoint`s of a `Pump` must have compatible media.
[nested-node] field=inlet_connection_point: s223: A `Pump` shall have at least one inlet using the medium `Fluid-Water`, `Fluid-Oil` or `Constituent-Refrigerant`.
[sparql] field=inlet_connection_point: The non-electrical `ConnectionPoint`s of a `Pump` must have compatible media.


## 8. The generated + hand-written override pattern

`semantic_objects.s223.properties` doesn't just re-export the generated
`QuantifiableObservableProperty` - it *extends* it with `qk`/`value`/`unit` fields
that aren't derivable from the s223 namespace this pipeline ingests (they come from
the QUDT namespace, out of scope for this pilot). Concrete quantity-kind classes like
`Area`/`Power`/`Tilt` are hand-written on top of that extension.

This is the pattern to follow for your own customizations: subclass the generated
base in the sibling non-generated module, never edit `_generated/*.py` directly (it's
overwritten on every regeneration).

In [16]:
area = properties.Area(150.0)
print(f"{area.value} {area.unit._name}, quantity kind: {area.qk._name}")
print()
print("Area's real base class (generated, no qk/value/unit):",
      properties.Area.__mro__[2].__module__ + "." + properties.Area.__mro__[2].__name__)

150.0 M2, quantity kind: Area

Area's real base class (generated, no qk/value/unit): semantic_objects.s223._generated.properties.QuantifiableObservableProperty


## Next steps

- `ontology-ingestion-tutorial.ipynb` - how the generation pipeline itself works, and
  how to extend it to another ontology.
- `docs/inference.md` - using SHACL-based type inference over raw RDF data.
- `src/semantic_objects/s223/_generated/_meta.py` - generation metadata (source
  checksum, quantity kinds referenced by the ontology, unresolved shapes).